In [1]:
# Import essential libraries

import pandas as pd
import numpy as np
from pathlib import Path

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Define cleaned dataset location
data_path = Path(
    "processed_data/CICIDS2017_cleaned.parquet"
)

# Load cleaned dataset
df = pd.read_parquet(data_path)

print("Cleaned dataset loaded successfully!")
print("-" * 40)

print("Dataset shape:", df.shape)
print("Total records:", len(df))
print("Total columns:", len(df.columns))
print("Input features:", df.shape[1] - 1)
print("Traffic categories:", df["Label"].nunique())

print("\nMemory usage (GB):",
      round(df.memory_usage(deep=True).sum()
            / 1024**3, 2))

display(df.head())

Cleaned dataset loaded successfully!
----------------------------------------
Dataset shape: (2520798, 71)
Total records: 2520798
Total columns: 71
Input features: 70
Traffic categories: 15

Memory usage (GB): 1.35


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [2]:
# Step 18B: Feature inspection

print("FEATURE INSPECTION")
print("-" * 50)

feature_columns = [
    col for col in df.columns
    if col != "Label"
]

print("Total input features:", len(feature_columns))

print("\nComplete list of input features:")

for i, feature in enumerate(feature_columns, start=1):
    print(f"{i}. {feature}")

print("\nPotential identifier or leakage-related features:")

keywords = [
    "ip", "timestamp", "flow id",
    "port", "label", "attack"
]

potential_leakage = [
    col for col in feature_columns
    if any(
        word in col.lower().split()
        or word == col.lower()
        for word in keywords
    )
]

print(potential_leakage)

print("\nDuplicate column names:",
      df.columns.duplicated().sum())

print("\nTarget column:", "Label")

FEATURE INSPECTION
--------------------------------------------------
Total input features: 70

Complete list of input features:
1. Destination Port
2. Flow Duration
3. Total Fwd Packets
4. Total Backward Packets
5. Total Length of Fwd Packets
6. Total Length of Bwd Packets
7. Fwd Packet Length Max
8. Fwd Packet Length Min
9. Fwd Packet Length Mean
10. Fwd Packet Length Std
11. Bwd Packet Length Max
12. Bwd Packet Length Min
13. Bwd Packet Length Mean
14. Bwd Packet Length Std
15. Flow Bytes/s
16. Flow Packets/s
17. Flow IAT Mean
18. Flow IAT Std
19. Flow IAT Max
20. Flow IAT Min
21. Fwd IAT Total
22. Fwd IAT Mean
23. Fwd IAT Std
24. Fwd IAT Max
25. Fwd IAT Min
26. Bwd IAT Total
27. Bwd IAT Mean
28. Bwd IAT Std
29. Bwd IAT Max
30. Bwd IAT Min
31. Fwd PSH Flags
32. Fwd URG Flags
33. Fwd Header Length
34. Bwd Header Length
35. Fwd Packets/s
36. Bwd Packets/s
37. Min Packet Length
38. Max Packet Length
39. Packet Length Mean
40. Packet Length Std
41. Packet Length Variance
42. FIN Flag Co

In [3]:
# Step 18C: Verify potentially duplicated features

print("DUPLICATE FEATURE VERIFICATION")
print("-" * 50)

feature_1 = "Fwd Header Length"
feature_2 = "Fwd Header Length.1"

if feature_1 in df.columns and feature_2 in df.columns:

    identical = df[feature_1].eq(df[feature_2]).all()

    differences = df[feature_1].ne(df[feature_2]).sum()

    print("First feature:", feature_1)
    print("Second feature:", feature_2)

    print("\nFeatures are identical:", identical)
    print("Number of differing records:", differences)

    print("\nSample comparison:")
    display(df[[feature_1, feature_2]].head(10))

else:
    print("One or both features are unavailable.")

DUPLICATE FEATURE VERIFICATION
--------------------------------------------------
First feature: Fwd Header Length
Second feature: Fwd Header Length.1

Features are identical: True
Number of differing records: 0

Sample comparison:


,Fwd Header Length,Fwd Header Length.1
0,40,40
1,20,20
2,20,20
3,20,20
4,40,40
5,40,40
6,40,40
7,20,20
8,40,40
9,40,40


In [4]:
# Step 18D: Remove the verified duplicate feature

before_columns = df.shape[1]

df = df.drop(columns=["Fwd Header Length.1"])

after_columns = df.shape[1]

print("DUPLICATE FEATURE REMOVAL")
print("-" * 50)

print("Columns before:", before_columns)
print("Columns after:", after_columns)
print("Features removed:", before_columns - after_columns)

print("\nUpdated dataset shape:", df.shape)
print("Remaining input features:", df.shape[1] - 1)

print("\nOriginal feature preserved:",
      "Fwd Header Length" in df.columns)

print("Duplicate feature removed:",
      "Fwd Header Length.1" not in df.columns)

print("Target column preserved:",
      "Label" in df.columns)

DUPLICATE FEATURE REMOVAL
--------------------------------------------------
Columns before: 71
Columns after: 70
Features removed: 1

Updated dataset shape: (2520798, 70)
Remaining input features: 69

Original feature preserved: True
Duplicate feature removed: True
Target column preserved: True


In [5]:
# Step 18E: Verify feature data types

import numpy as np

input_features = df.columns.drop("Label")

numeric_features = df[input_features].select_dtypes(
    include=[np.number]
).columns.tolist()

non_numeric_features = [
    col for col in input_features
    if col not in numeric_features
]

print("FEATURE DATA TYPE VERIFICATION")
print("-" * 50)

print("Total input features:", len(input_features))
print("Numerical features:", len(numeric_features))
print("Non-numerical features:", len(non_numeric_features))

print("\nNon-numerical feature names:")
print(non_numeric_features)

print("\nTarget column data type:")
print(df["Label"].dtype)

print("\nInput feature data type distribution:")
print(df.dtypes.drop("Label").value_counts())

print("\nAll input features are numerical:",
      len(non_numeric_features) == 0)

FEATURE DATA TYPE VERIFICATION
--------------------------------------------------
Total input features: 69
Numerical features: 69
Non-numerical features: 0

Non-numerical feature names:
[]

Target column data type:
str

Input feature data type distribution:
int64      45
float64    24
Name: count, dtype: int64

All input features are numerical: True


In [6]:
# Step 18F: Target label encoding

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(df["Label"])

print("TARGET LABEL ENCODING")
print("-" * 50)

print("Total traffic categories:",
      len(label_encoder.classes_))

print("\nLabel encoding mapping:")

for index, label in enumerate(label_encoder.classes_):
    print(f"{index}: {label}")

print("\nTotal encoded records:", len(y))

print("Encoded target data type:", y.dtype)

print("Original target preserved:",
      "Label" in df.columns)

print("\nFirst 10 encoded labels:")
print(y[:10])

TARGET LABEL ENCODING
--------------------------------------------------
Total traffic categories: 15

Label encoding mapping:
0: BENIGN
1: Bot
2: DDoS
3: DoS GoldenEye
4: DoS Hulk
5: DoS Slowhttptest
6: DoS slowloris
7: FTP-Patator
8: Heartbleed
9: Infiltration
10: PortScan
11: SSH-Patator
12: Web Attack - Brute Force
13: Web Attack - SQL Injection
14: Web Attack - XSS

Total encoded records: 2520798
Encoded target data type: int64
Original target preserved: True

First 10 encoded labels:
[0 0 0 0 0 0 0 0 0 0]


In [7]:
# Step 18G: Save label encoding mapping

import pandas as pd
from pathlib import Path

label_mapping = pd.DataFrame({
    "Encoded_Label": range(len(label_encoder.classes_)),
    "Traffic_Category": label_encoder.classes_
})

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

mapping_path = results_dir / "02_label_encoding_mapping.csv"

label_mapping.to_csv(mapping_path, index=False)

print("LABEL MAPPING VERIFICATION")
print("-" * 50)

display(label_mapping)

print("\nMapping saved successfully:",
      mapping_path.exists())

print("Total categories saved:", len(label_mapping))

print("File location:", mapping_path.resolve())

LABEL MAPPING VERIFICATION
--------------------------------------------------


,Encoded_Label,Traffic_Category
0,0,BENIGN
1,1,Bot
2,2,DDoS
3,3,DoS GoldenEye
4,4,DoS Hulk
5,5,DoS Slowhttptest
6,6,DoS slowloris
7,7,FTP-Patator
8,8,Heartbleed
9,9,Infiltration



Mapping saved successfully: True
Total categories saved: 15
File location: C:\Users\pc\XAI-Intrusion-Detection\results\02_label_encoding_mapping.csv


In [8]:
# Step 18H: Feature and target separation

X = df.drop(columns=["Label"])

print("FEATURE AND TARGET SEPARATION")
print("-" * 50)

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

print("\nTotal input features:", X.shape[1])
print("Total feature records:", len(X))
print("Total target records:", len(y))

print("\nTarget excluded from features:",
      "Label" not in X.columns)

print("Feature and target alignment:",
      len(X) == len(y))

print("\nSample input features:")
display(X.head())

FEATURE AND TARGET SEPARATION
--------------------------------------------------
Feature matrix shape: (2520798, 69)
Target vector shape: (2520798,)

Total input features: 69
Total feature records: 2520798
Total target records: 2520798

Target excluded from features: True
Feature and target alignment: True

Sample input features:


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,1,20,0.0,0.0,0,0,0.0,0.0,0,0
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,0,20,0.0,0.0,0,0,0.0,0.0,0,0
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,0,20,0.0,0.0,0,0,0.0,0.0,0,0
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,0,20,0.0,0.0,0,0,0.0,0.0,0,0
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,1,20,0.0,0.0,0,0,0.0,0.0,0,0


In [9]:
# Step 18I: Stratified training and testing split

import numpy as np
from sklearn.model_selection import train_test_split

indices = np.arange(len(y))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=y
)

y_train = y[train_idx]
y_test = y[test_idx]

print("STRATIFIED DATASET SPLIT")
print("-" * 50)

print("Total records:", len(y))
print("Training records:", len(train_idx))
print("Testing records:", len(test_idx))

print("\nTraining percentage:",
      round(len(train_idx) / len(y) * 100, 2))

print("Testing percentage:",
      round(len(test_idx) / len(y) * 100, 2))

print("\nTraining categories:",
      len(np.unique(y_train)))

print("Testing categories:",
      len(np.unique(y_test)))

print("\nOverlapping row indices:",
      len(np.intersect1d(train_idx, test_idx)))

print("All records accounted for:",
      len(train_idx) + len(test_idx) == len(y))

STRATIFIED DATASET SPLIT
--------------------------------------------------
Total records: 2520798
Training records: 2016638
Testing records: 504160

Training percentage: 80.0
Testing percentage: 20.0

Training categories: 15
Testing categories: 15

Overlapping row indices: 0
All records accounted for: True


In [10]:
# Step 18J: Verify class distribution

class_distribution = pd.DataFrame({
    "Traffic_Category": label_encoder.classes_,
    "Total_Records": np.bincount(y, minlength=15),
    "Training_Records": np.bincount(y_train, minlength=15),
    "Testing_Records": np.bincount(y_test, minlength=15)
})

class_distribution["Training_Percentage"] = (
    class_distribution["Training_Records"] /
    class_distribution["Total_Records"] * 100
).round(2)

class_distribution["Testing_Percentage"] = (
    class_distribution["Testing_Records"] /
    class_distribution["Total_Records"] * 100
).round(2)

print("CLASS DISTRIBUTION VERIFICATION")
print("-" * 50)

display(class_distribution)

print("\nAll categories represented in training:",
      (class_distribution["Training_Records"] > 0).all())

print("All categories represented in testing:",
      (class_distribution["Testing_Records"] > 0).all())

distribution_path = results_dir / "02_train_test_distribution.csv"

class_distribution.to_csv(distribution_path, index=False)

print("\nDistribution saved successfully:",
      distribution_path.exists())


CLASS DISTRIBUTION VERIFICATION
--------------------------------------------------


,Traffic_Category,Total_Records,Training_Records,Testing_Records,Training_Percentage,Testing_Percentage
0,BENIGN,2095057,1676045,419012,80.00,20.00
1,Bot,1948,1558,390,79.98,20.02
2,DDoS,128014,102411,25603,80.00,20.00
3,DoS GoldenEye,10286,8229,2057,80.00,20.00
4,DoS Hulk,172846,138277,34569,80.00,20.00
5,DoS Slowhttptest,5228,4182,1046,79.99,20.01
6,DoS slowloris,5385,4308,1077,80.00,20.00
7,FTP-Patator,5931,4745,1186,80.00,20.00
8,Heartbleed,11,9,2,81.82,18.18
9,Infiltration,36,29,7,80.56,19.44



All categories represented in training: True
All categories represented in testing: True

Distribution saved successfully: True


In [11]:
# Step 18K: Calculate balanced class weights

from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(classes, weights)
}

class_weight_df = pd.DataFrame({
    "Encoded_Label": classes,
    "Traffic_Category": label_encoder.inverse_transform(classes),
    "Class_Weight": weights
})

print("CLASS WEIGHT CALCULATION")
print("-" * 50)

display(class_weight_df.round(4))

print("\nTotal class weights:", len(class_weights))

print("All weights are positive:",
      all(weight > 0 for weight in class_weights.values()))

print("Calculated using training data only: True")

weight_path = results_dir / "02_class_weights.csv"

class_weight_df.to_csv(weight_path, index=False)

print("\nClass weights saved successfully:",
      weight_path.exists())

CLASS WEIGHT CALCULATION
--------------------------------------------------


,Encoded_Label,Traffic_Category,Class_Weight
0,0,BENIGN,0.0802
1,1,Bot,86.2917
2,2,DDoS,1.3128
3,3,DoS GoldenEye,16.3377
4,4,DoS Hulk,0.9723
5,5,DoS Slowhttptest,32.1479
6,6,DoS slowloris,31.2076
7,7,FTP-Patator,28.3335
8,8,Heartbleed,14938.0593
9,9,Infiltration,4635.9494



Total class weights: 15
All weights are positive: True
Calculated using training data only: True

Class weights saved successfully: True


In [12]:
# Step 18L: Feature scaling requirement analysis

feature_statistics = []

for column in X.columns:
    values = X[column].iloc[train_idx]

    feature_statistics.append({
        "Feature": column,
        "Minimum": values.min(),
        "Maximum": values.max(),
        "Mean": values.mean(),
        "Standard_Deviation": values.std()
    })

feature_scale_df = pd.DataFrame(feature_statistics)

feature_scale_df["Range"] = (
    feature_scale_df["Maximum"] -
    feature_scale_df["Minimum"]
)

print("FEATURE SCALING REQUIREMENT ANALYSIS")
print("-" * 55)

print("Total features analysed:", len(feature_scale_df))

print("\nTop 15 features with the largest numerical ranges:")

display(
    feature_scale_df.sort_values(
        "Range", ascending=False
    ).head(15)
)

scale_path = results_dir / "02_feature_scale_analysis.csv"

feature_scale_df.to_csv(scale_path, index=False)

print("\nFeature scale analysis saved successfully:",
      scale_path.exists())

FEATURE SCALING REQUIREMENT ANALYSIS
-------------------------------------------------------
Total features analysed: 69

Top 15 features with the largest numerical ranges:


,Feature,Minimum,Maximum,Mean,Standard_Deviation,Range
32,Fwd Header Length,-9.663668e+09,4.617240e+06,-1.569985e+04,9.240046e+06,9.668285e+09
14,Flow Bytes/s,-2.610000e+08,2.071000e+09,1.397915e+06,2.631550e+07,2.332000e+09
33,Bwd Header Length,-1.073741e+09,5.838440e+06,-2.106348e+03,1.345290e+06,1.079580e+09
5,Total Length of Bwd Packets,0.000000e+00,6.554530e+08,1.904944e+04,2.487402e+06,6.554530e+08
56,Subflow Bwd Bytes,0.000000e+00,6.554530e+08,1.904938e+04,2.487423e+06,6.554530e+08
60,min_seg_size_forward,-5.368707e+08,1.380000e+02,-2.627997e+03,1.032847e+06,5.368708e+08
19,Flow IAT Min,-1.400000e+01,1.200000e+08,1.723256e+05,3.036265e+06,1.200000e+08
16,Flow IAT Mean,-1.200000e+01,1.200000e+08,1.448299e+06,4.696130e+06,1.200000e+08
24,Fwd IAT Min,-1.200000e+01,1.200000e+08,1.138651e+06,9.071585e+06,1.200000e+08
18,Flow IAT Max,-1.200000e+01,1.200000e+08,1.030094e+07,2.568308e+07,1.200000e+08



Feature scale analysis saved successfully: True


In [13]:
# Step 18M: Negative value investigation

negative_statistics = []

for column in X.columns:
    values = X[column].iloc[train_idx]

    negative_count = int((values < 0).sum())

    if negative_count > 0:
        negative_statistics.append({
            "Feature": column,
            "Minimum_Value": values.min(),
            "Negative_Records": negative_count,
            "Negative_Percentage": (
                negative_count / len(values)
            ) * 100
        })

negative_df = pd.DataFrame(
    negative_statistics,
    columns=[
        "Feature",
        "Minimum_Value",
        "Negative_Records",
        "Negative_Percentage"
    ]
)

negative_df = negative_df.sort_values(
    "Negative_Records",
    ascending=False
)

print("NEGATIVE VALUE INVESTIGATION")
print("-" * 55)

print("Total features investigated:", X.shape[1])

print("Features containing negative values:",
      len(negative_df))

display(negative_df.round(6))

negative_path = (
    results_dir / "02_negative_value_analysis.csv"
)

negative_df.to_csv(negative_path, index=False)

print("\nAnalysis saved successfully:",
      negative_path.exists())

NEGATIVE VALUE INVESTIGATION
-------------------------------------------------------
Total features investigated: 69
Features containing negative values: 12


,Feature,Minimum_Value,Negative_Records,Negative_Percentage
10,Init_Win_bytes_backward,-1.000000e+00,972519,48.224768
9,Init_Win_bytes_forward,-1.000000e+00,728727,36.135737
5,Flow IAT Min,-1.400000e+01,2305,0.114299
0,Flow Duration,-1.200000e+01,96,0.004760
2,Flow Packets/s,-2.000000e+06,96,0.004760
3,Flow IAT Mean,-1.200000e+01,96,0.004760
4,Flow IAT Max,-1.200000e+01,96,0.004760
1,Flow Bytes/s,-2.610000e+08,70,0.003471
11,min_seg_size_forward,-5.368707e+08,26,0.001289
7,Fwd Header Length,-9.663668e+09,26,0.001289



Analysis saved successfully: True


In [14]:
# Step 18N: Invalid record impact assessment

invalid_features = [
    "Flow IAT Min",
    "Flow Duration",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Max",
    "Flow Bytes/s",
    "min_seg_size_forward",
    "Fwd Header Length",
    "Bwd Header Length",
    "Fwd IAT Min"
]

invalid_mask = np.zeros(len(train_idx), dtype=bool)

for feature in invalid_features:
    invalid_mask |= (
        X[feature].iloc[train_idx].to_numpy() < 0
    )

affected_count = int(invalid_mask.sum())

training_labels = np.asarray(y)[train_idx]

impact_records = []

for label in np.unique(training_labels):
    class_mask = training_labels == label

    total = int(class_mask.sum())

    affected = int(
        (class_mask & invalid_mask).sum()
    )

    impact_records.append({
        "Encoded_Label": label,
        "Traffic_Category": label_encoder.classes_[label],
        "Training_Records": total,
        "Affected_Records": affected,
        "Affected_Percentage": affected / total * 100
    })

impact_df = pd.DataFrame(impact_records)

print("INVALID RECORD IMPACT ASSESSMENT")
print("-" * 55)

print("Features investigated:", len(invalid_features))
print("Total training records:", len(train_idx))
print("Unique affected records:", affected_count)

print("Affected percentage:",
      round(affected_count / len(train_idx) * 100, 6))

display(impact_df.round(6))

impact_path = (
    results_dir / "02_invalid_record_impact.csv"
)

impact_df.to_csv(impact_path, index=False)

print("\nImpact analysis saved successfully:",
      impact_path.exists())

INVALID RECORD IMPACT ASSESSMENT
-------------------------------------------------------
Features investigated: 10
Total training records: 2016638
Unique affected records: 2331
Affected percentage: 0.115588


,Encoded_Label,Traffic_Category,Training_Records,Affected_Records,Affected_Percentage
0,0,BENIGN,1676045,2168,0.129352
1,1,Bot,1558,0,0.000000
2,2,DDoS,102411,16,0.015623
3,3,DoS GoldenEye,8229,5,0.060761
4,4,DoS Hulk,138277,133,0.096184
5,5,DoS Slowhttptest,4182,0,0.000000
6,6,DoS slowloris,4308,0,0.000000
7,7,FTP-Patator,4745,2,0.042150
8,8,Heartbleed,9,4,44.444444
9,9,Infiltration,29,1,3.448276



Impact analysis saved successfully: True


In [15]:
# Step 18O: Training-based median calculation

replacement_values = {}
median_records = []

for feature in invalid_features:

    training_values = (
        X[feature].iloc[train_idx].to_numpy()
    )

    valid_values = training_values[
        (training_values >= 0) &
        np.isfinite(training_values)
    ]

    median_value = float(np.median(valid_values))

    replacement_values[feature] = median_value

    median_records.append({
        "Feature": feature,
        "Valid_Training_Records": len(valid_values),
        "Negative_Training_Records": int(
            (training_values < 0).sum()
        ),
        "Replacement_Median": median_value
    })

median_df = pd.DataFrame(median_records)

print("TRAINING-BASED MEDIAN CALCULATION")
print("-" * 55)

print("Total features processed:",
      len(replacement_values))

display(median_df)

print("\nAll replacement values are finite:",
      all(np.isfinite(v)
          for v in replacement_values.values()))

print("All replacement values are non-negative:",
      all(v >= 0
          for v in replacement_values.values()))

median_path = (
    results_dir / "02_median_replacement_values.csv"
)

median_df.to_csv(median_path, index=False)

print("\nReplacement values saved successfully:",
      median_path.exists())

TRAINING-BASED MEDIAN CALCULATION
-------------------------------------------------------
Total features processed: 10


,Feature,Valid_Training_Records,Negative_Training_Records,Replacement_Median
0,Flow IAT Min,2014333,2305,4.000000
1,Flow Duration,2016542,96,50670.000000
2,Flow Packets/s,2016542,96,69.391437
3,Flow IAT Mean,2016542,96,20455.333330
4,Flow IAT Max,2016542,96,46626.000000
5,Flow Bytes/s,2016568,70,3714.316190
6,min_seg_size_forward,2016612,26,20.000000
7,Fwd Header Length,2016612,26,64.000000
8,Bwd Header Length,2016620,18,40.000000
9,Fwd IAT Min,2016623,15,3.000000



All replacement values are finite: True
All replacement values are non-negative: True

Replacement values saved successfully: True


In [16]:
# Step 18P: Invalid value treatment

treatment_records = []

for feature, median_value in replacement_values.items():

    values = X[feature].to_numpy(
        dtype=np.float64, copy=True
    )

    negative_mask = values < 0

    training_replacements = int(
        negative_mask[train_idx].sum()
    )

    testing_replacements = int(
        negative_mask[test_idx].sum()
    )

    total_replacements = int(
        negative_mask.sum()
    )

    values[negative_mask] = median_value

    X[feature] = values

    treatment_records.append({
        "Feature": feature,
        "Replacement_Median": median_value,
        "Training_Replacements": training_replacements,
        "Testing_Replacements": testing_replacements,
        "Total_Replacements": total_replacements,
        "Remaining_Negative_Values": int(
            (values < 0).sum()
        )
    })

treatment_df = pd.DataFrame(treatment_records)

print("INVALID VALUE TREATMENT")
print("-" * 60)

display(treatment_df)

print("\nTotal features treated:",
      len(treatment_df))

print("Total training values replaced:",
      treatment_df["Training_Replacements"].sum())

print("Total testing values replaced:",
      treatment_df["Testing_Replacements"].sum())

print("Total values replaced:",
      treatment_df["Total_Replacements"].sum())

print("\nRemaining invalid negative values:",
      treatment_df["Remaining_Negative_Values"].sum())

print("Updated feature matrix shape:", X.shape)

print("All records preserved:",
      len(X) == len(y))

treatment_path = (
    results_dir / "02_invalid_value_treatment.csv"
)

treatment_df.to_csv(treatment_path, index=False)

print("\nTreatment report saved successfully:",
      treatment_path.exists())


INVALID VALUE TREATMENT
------------------------------------------------------------


,Feature,Replacement_Median,Training_Replacements,Testing_Replacements,Total_Replacements,Remaining_Negative_Values
0,Flow IAT Min,4.000000,2305,575,2880,0
1,Flow Duration,50670.000000,96,11,107,0
2,Flow Packets/s,69.391437,96,11,107,0
3,Flow IAT Mean,20455.333330,96,11,107,0
4,Flow IAT Max,46626.000000,96,11,107,0
5,Flow Bytes/s,3714.316190,70,8,78,0
6,min_seg_size_forward,20.000000,26,9,35,0
7,Fwd Header Length,64.000000,26,9,35,0
8,Bwd Header Length,40.000000,18,4,22,0
9,Fwd IAT Min,3.000000,15,2,17,0



Total features treated: 10
Total training values replaced: 2844
Total testing values replaced: 651
Total values replaced: 3495

Remaining invalid negative values: 0
Updated feature matrix shape: (2520798, 69)
All records preserved: True

Treatment report saved successfully: True


In [17]:
# Step 18Q: Post-treatment data quality verification

sentinel_features = [
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward"
]

quality_records = []

for feature in X.columns:

    values = X[feature].to_numpy()

    missing_count = int(pd.isna(values).sum())

    infinite_count = int(np.isinf(values).sum())

    negative_count = int((values < 0).sum())

    if feature in sentinel_features:

        sentinel_count = int((values == -1).sum())

        unexpected_negative = int(
            ((values < 0) & (values != -1)).sum()
        )

    else:

        sentinel_count = 0
        unexpected_negative = negative_count

    quality_records.append({
        "Feature": feature,
        "Missing_Values": missing_count,
        "Infinite_Values": infinite_count,
        "Negative_Values": negative_count,
        "Retained_Sentinel_Values": sentinel_count,
        "Unexpected_Negative_Values": unexpected_negative
    })

quality_df = pd.DataFrame(quality_records)

print("POST-TREATMENT DATA QUALITY VERIFICATION")
print("-" * 60)

print("Total features verified:", len(quality_df))

print("Total records:", len(X))

print("\nTotal missing values:",
      quality_df["Missing_Values"].sum())

print("Total infinite values:",
      quality_df["Infinite_Values"].sum())

print("Total unexpected negative values:",
      quality_df["Unexpected_Negative_Values"].sum())

print("Total retained sentinel values:",
      quality_df["Retained_Sentinel_Values"].sum())

print("\nTCP window sentinel verification:")

display(
    quality_df[
        quality_df["Feature"].isin(sentinel_features)
    ]
)

print("\nFeature matrix shape:", X.shape)

print("All records preserved:",
      len(X) == 2520798)

print("All input features preserved:",
      X.shape[1] == 69)

print("Feature and target lengths match:",
      len(X) == len(y))

quality_path = (
    results_dir / "02_post_treatment_quality.csv"
)

quality_df.to_csv(quality_path, index=False)

print("\nQuality verification saved successfully:",
      quality_path.exists())

POST-TREATMENT DATA QUALITY VERIFICATION
------------------------------------------------------------
Total features verified: 69
Total records: 2520798

Total missing values: 0
Total infinite values: 0
Total unexpected negative values: 0
Total retained sentinel values: 2126637

TCP window sentinel verification:


,Feature,Missing_Values,Infinite_Values,Negative_Values,Retained_Sentinel_Values,Unexpected_Negative_Values
57,Init_Win_bytes_forward,0,0,911012,911012,0
58,Init_Win_bytes_backward,0,0,1215625,1215625,0



Feature matrix shape: (2520798, 69)
All records preserved: True
All input features preserved: True
Feature and target lengths match: True

Quality verification saved successfully: True


In [18]:
# Step 18R: Training-based feature scaler fitting

from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()

batch_size = 100000

print("TRAINING-BASED FEATURE SCALER FITTING")
print("-" * 60)

print("Training records:", len(train_idx))
print("Input features:", X.shape[1])
print("Batch size:", batch_size)

for start in range(0, len(train_idx), batch_size):

    end = min(start + batch_size, len(train_idx))

    batch_indices = train_idx[start:end]

    training_batch = X.iloc[batch_indices]

    scaler.partial_fit(training_batch)

    print(f"Processed: {end:,} / {len(train_idx):,}")

del training_batch

print("\nSCALER FITTING VERIFICATION")
print("-" * 60)

print("Features fitted:", scaler.n_features_in_)

print("Training samples processed:",
      scaler.n_samples_seen_)

print("All training samples processed:",
      np.all(scaler.n_samples_seen_ == len(train_idx)))

print("All means are finite:",
      np.isfinite(scaler.mean_).all())

print("All scaling factors are finite:",
      np.isfinite(scaler.scale_).all())

print("All scaling factors are positive:",
      (scaler.scale_ > 0).all())

scaler_path = results_dir / "02_standard_scaler.joblib"

joblib.dump(scaler, scaler_path)

print("\nScaler saved successfully:",
      scaler_path.exists())

TRAINING-BASED FEATURE SCALER FITTING
------------------------------------------------------------
Training records: 2016638
Input features: 69
Batch size: 100000
Processed: 100,000 / 2,016,638
Processed: 200,000 / 2,016,638
Processed: 300,000 / 2,016,638
Processed: 400,000 / 2,016,638
Processed: 500,000 / 2,016,638
Processed: 600,000 / 2,016,638
Processed: 700,000 / 2,016,638
Processed: 800,000 / 2,016,638
Processed: 900,000 / 2,016,638
Processed: 1,000,000 / 2,016,638
Processed: 1,100,000 / 2,016,638
Processed: 1,200,000 / 2,016,638
Processed: 1,300,000 / 2,016,638
Processed: 1,400,000 / 2,016,638
Processed: 1,500,000 / 2,016,638
Processed: 1,600,000 / 2,016,638
Processed: 1,700,000 / 2,016,638
Processed: 1,800,000 / 2,016,638
Processed: 1,900,000 / 2,016,638
Processed: 2,000,000 / 2,016,638
Processed: 2,016,638 / 2,016,638

SCALER FITTING VERIFICATION
------------------------------------------------------------
Features fitted: 69
Training samples processed: 2016638.0
All training s

### 18S. Scaler Parameter Verification

The fitted StandardScaler parameters are examined
to verify the numerical validity of the learned
feature means, variances and scaling factors.

The parameters are derived exclusively from the
training dataset and saved for reproducibility,
transparency and subsequent model development.

The original feature names and ordering are
preserved to support explainable AI analysis.

In [19]:
# Step 18S: Scaler parameter verification

scaler_df = pd.DataFrame({
    "Feature": X.columns,
    "Training_Mean": scaler.mean_,
    "Training_Variance": scaler.var_,
    "Scaling_Factor": scaler.scale_
})

scaler_df["Constant_Feature"] = (
    scaler_df["Training_Variance"] == 0
)

print("SCALER PARAMETER VERIFICATION")
print("-" * 60)

print("Total features:", len(scaler_df))

print("Finite means:",
      np.isfinite(scaler.mean_).all())

print("Finite variances:",
      np.isfinite(scaler.var_).all())

print("Non-negative variances:",
      (scaler.var_ >= 0).all())

print("Positive scaling factors:",
      (scaler.scale_ > 0).all())

print("Constant features:",
      scaler_df["Constant_Feature"].sum())

print("Feature ordering preserved:",
      np.array_equal(
          scaler.feature_names_in_,
          X.columns.to_numpy()
      ))

print("\nSample scaler parameters:")

display(scaler_df.head(15))

parameter_path = (
    results_dir / "02_scaler_parameters.csv"
)

scaler_df.to_csv(parameter_path, index=False)

print("\nScaler parameters saved successfully:",
      parameter_path.exists())


SCALER PARAMETER VERIFICATION
------------------------------------------------------------
Total features: 69
Finite means: True
Finite variances: True
Non-negative variances: True
Positive scaling factors: True
Constant features: 0
Feature ordering preserved: True

Sample scaler parameters:


,Feature,Training_Mean,Training_Variance,Scaling_Factor,Constant_Feature
0,Destination Port,8.684874e+03,3.612707e+08,1.900712e+04,False
1,Flow Duration,1.659789e+07,1.241495e+15,3.523486e+07,False
2,Total Fwd Packets,1.057677e+01,6.783876e+05,8.236429e+02,False
3,Total Backward Packets,1.196931e+01,1.205220e+06,1.097825e+03,False
4,Total Length of Fwd Packets,6.051642e+02,4.154644e+07,6.445653e+03,False
5,Total Length of Bwd Packets,1.904944e+04,6.187168e+12,2.487402e+06,False
6,Fwd Packet Length Max,2.310964e+02,5.698138e+05,7.548601e+02,False
7,Fwd Packet Length Min,1.921776e+01,3.708445e+03,6.089700e+01,False
8,Fwd Packet Length Mean,6.343593e+01,3.804404e+04,1.950488e+02,False
9,Fwd Packet Length Std,7.725991e+01,8.781655e+04,2.963386e+02,False



Scaler parameters saved successfully: True


### 18T. Dataset Transformation

The fitted StandardScaler is applied to both
training and testing datasets using the parameters
learned exclusively from the training data.

Batch-based transformation is employed to reduce
memory consumption when processing the large
network traffic dataset.

The transformed datasets are stored in NumPy
format using 32-bit floating-point precision.

The original unscaled features are preserved
for tree-based machine learning models and
subsequent explainability analysis.

No additional scaler fitting is performed on
the testing dataset.

In [20]:
# Step 18T: Batch-based dataset transformation

from numpy.lib.format import open_memmap

batch_size = 100000

train_scaled_path = (
    results_dir / "02_X_train_scaled.npy"
)

test_scaled_path = (
    results_dir / "02_X_test_scaled.npy"
)

print("BATCH-BASED DATASET TRANSFORMATION")
print("-" * 60)

def transform_dataset(indices, output_path, name):

    output = open_memmap(
        output_path,
        mode="w+",
        dtype=np.float32,
        shape=(len(indices), X.shape[1])
    )

    print(f"\nTransforming {name} dataset:")

    for start in range(0, len(indices), batch_size):

        end = min(start + batch_size, len(indices))

        batch_indices = indices[start:end]

        batch = X.iloc[batch_indices]

        transformed = scaler.transform(batch)

        output[start:end] = transformed.astype(
            np.float32
        )

        print(f"Processed: {end:,} / {len(indices):,}")

    output.flush()

    del output

    print(f"{name} transformation completed.")


transform_dataset(
    train_idx,
    train_scaled_path,
    "Training"
)

transform_dataset(
    test_idx,
    test_scaled_path,
    "Testing"
)

print("\nTRANSFORMATION VERIFICATION")
print("-" * 60)

X_train_scaled = np.load(
    train_scaled_path,
    mmap_mode="r"
)

X_test_scaled = np.load(
    test_scaled_path,
    mmap_mode="r"
)

print("Training shape:", X_train_scaled.shape)

print("Testing shape:", X_test_scaled.shape)

print("Training data type:", X_train_scaled.dtype)

print("Testing data type:", X_test_scaled.dtype)

print("Training file saved:",
      train_scaled_path.exists())

print("Testing file saved:",
      test_scaled_path.exists())

print("Original feature matrix preserved:",
      X.shape == (2520798, 69))

BATCH-BASED DATASET TRANSFORMATION
------------------------------------------------------------

Transforming Training dataset:
Processed: 100,000 / 2,016,638
Processed: 200,000 / 2,016,638
Processed: 300,000 / 2,016,638
Processed: 400,000 / 2,016,638
Processed: 500,000 / 2,016,638
Processed: 600,000 / 2,016,638
Processed: 700,000 / 2,016,638
Processed: 800,000 / 2,016,638
Processed: 900,000 / 2,016,638
Processed: 1,000,000 / 2,016,638
Processed: 1,100,000 / 2,016,638
Processed: 1,200,000 / 2,016,638
Processed: 1,300,000 / 2,016,638
Processed: 1,400,000 / 2,016,638
Processed: 1,500,000 / 2,016,638
Processed: 1,600,000 / 2,016,638
Processed: 1,700,000 / 2,016,638
Processed: 1,800,000 / 2,016,638
Processed: 1,900,000 / 2,016,638
Processed: 2,000,000 / 2,016,638
Processed: 2,016,638 / 2,016,638
Training transformation completed.

Transforming Testing dataset:
Processed: 100,000 / 504,160
Processed: 200,000 / 504,160
Processed: 300,000 / 504,160
Processed: 400,000 / 504,160
Processed: 500,

### 18U. Post-Transformation Numerical Verification

The transformed training and testing datasets
are examined to verify numerical integrity
following feature standardization.

The verification process checks for missing
and infinite values, dimensional consistency,
and the statistical properties of the
standardized training features.

Training feature means are expected to be
approximately zero, while standard deviations
are expected to be approximately one.

The testing dataset is evaluated independently
without requiring its statistical distribution
to match that of the training dataset.

Batch-based verification is employed to
minimize memory consumption.



In [21]:
# Step 18U: Post-transformation numerical verification

import numpy as np
import pandas as pd

batch_size = 100000

print("POST-TRANSFORMATION NUMERICAL VERIFICATION")
print("-" * 60)


def verify_scaled_dataset(data, name):

    n_features = data.shape[1]

    feature_sum = np.zeros(n_features, dtype=np.float64)

    feature_squared_sum = np.zeros(
        n_features, dtype=np.float64
    )

    nan_count = 0
    inf_count = 0

    for start in range(0, len(data), batch_size):

        end = min(start + batch_size, len(data))

        batch = np.asarray(
            data[start:end],
            dtype=np.float64
        )

        nan_count += np.isnan(batch).sum()

        inf_count += np.isinf(batch).sum()

        feature_sum += batch.sum(axis=0)

        feature_squared_sum += (
            batch ** 2
        ).sum(axis=0)

    means = feature_sum / len(data)

    variances = (
        feature_squared_sum / len(data)
    ) - means ** 2

    stds = np.sqrt(
        np.maximum(variances, 0)
    )

    print(f"\n{name} dataset")
    print("Records:", len(data))
    print("Features:", n_features)
    print("NaN values:", nan_count)
    print("Infinite values:", inf_count)
    print("All values finite:",
          nan_count == 0 and inf_count == 0)

    return means, stds, nan_count, inf_count


train_means, train_stds, train_nan, train_inf = (
    verify_scaled_dataset(
        X_train_scaled,
        "Training"
    )
)

test_means, test_stds, test_nan, test_inf = (
    verify_scaled_dataset(
        X_test_scaled,
        "Testing"
    )
)

verification_df = pd.DataFrame({
    "Feature": X.columns,
    "Training_Mean": train_means,
    "Training_Std": train_stds,
    "Testing_Mean": test_means,
    "Testing_Std": test_stds
})

print("\nSTANDARDIZATION VERIFICATION")
print("-" * 60)

print("Training means approximately zero:",
      np.allclose(train_means, 0, atol=1e-5))

print("Training standard deviations approximately one:",
      np.allclose(train_stds, 1, atol=1e-5))

print("Training shape correct:",
      X_train_scaled.shape == (2016638, 69))

print("Testing shape correct:",
      X_test_scaled.shape == (504160, 69))

print("\nSample feature statistics:")

display(verification_df.head(15))

verification_path = (
    results_dir /
    "02_scaled_data_verification.csv"
)

verification_df.to_csv(
    verification_path,
    index=False
)

print("\nVerification report saved successfully:",
      verification_path.exists())

POST-TRANSFORMATION NUMERICAL VERIFICATION
------------------------------------------------------------

Training dataset
Records: 2016638
Features: 69
NaN values: 0
Infinite values: 0
All values finite: True

Testing dataset
Records: 504160
Features: 69
NaN values: 0
Infinite values: 0
All values finite: True

STANDARDIZATION VERIFICATION
------------------------------------------------------------
Training means approximately zero: True
Training standard deviations approximately one: True
Training shape correct: True
Testing shape correct: True

Sample feature statistics:


,Feature,Training_Mean,Training_Std,Testing_Mean,Testing_Std
0,Destination Port,-2.278903e-09,1.0,0.001504,1.001491
1,Flow Duration,-1.072822e-10,1.0,-0.000890,0.999701
2,Total Fwd Packets,-6.589225e-11,1.0,-0.001791,0.807150
3,Total Backward Packets,-2.607805e-11,1.0,-0.001806,0.796464
4,Total Length of Fwd Packets,-2.874850e-10,1.0,0.005262,3.080952
5,Total Length of Bwd Packets,5.821556e-11,1.0,-0.001819,0.804813
6,Fwd Packet Length Max,-3.406394e-09,1.0,0.000880,1.009997
7,Fwd Packet Length Min,6.978507e-09,1.0,-0.001172,0.991871
8,Fwd Packet Length Mean,-1.931424e-10,1.0,0.001770,1.012846
9,Fwd Packet Length Std,5.491398e-09,1.0,0.001073,1.009124



Verification report saved successfully: True


### 18V. Scaled Dataset Persistence Verification

The saved standardized training and testing
datasets are independently reloaded from disk
to verify successful data persistence.

Memory-mapped loading is employed to minimize
memory consumption when accessing the large
NumPy datasets.

The reloaded datasets are examined to confirm
dimensional consistency, numerical precision,
and agreement with the transformed datasets.

The fitted scaler is also reloaded to verify
that its learned parameters remain available
for reproducible model development and
subsequent inference.

In [22]:
# Step 18V: Scaled dataset persistence verification

import numpy as np
import joblib

print("SCALED DATASET PERSISTENCE VERIFICATION")
print("-" * 60)

# Reload saved datasets using memory mapping

reloaded_train = np.load(
    train_scaled_path,
    mmap_mode="r"
)

reloaded_test = np.load(
    test_scaled_path,
    mmap_mode="r"
)

# Reload fitted scaler

reloaded_scaler = joblib.load(scaler_path)

print("\nRELOADED TRAINING DATASET")
print("Shape:", reloaded_train.shape)
print("Data type:", reloaded_train.dtype)
print("Shape correct:",
      reloaded_train.shape == (2016638, 69))

print("\nRELOADED TESTING DATASET")
print("Shape:", reloaded_test.shape)
print("Data type:", reloaded_test.dtype)
print("Shape correct:",
      reloaded_test.shape == (504160, 69))

# Verify saved dataset consistency in batches

def verify_persistence(original, reloaded):

    if original.shape != reloaded.shape:
        return False

    for start in range(0, len(original), 100000):

        end = min(start + 100000, len(original))

        if not np.array_equal(
            original[start:end],
            reloaded[start:end]
        ):
            return False

    return True


print("\nDATA CONSISTENCY VERIFICATION")
print("-" * 60)

print("Training values preserved:",
      verify_persistence(
          X_train_scaled,
          reloaded_train
      ))

print("Testing values preserved:",
      verify_persistence(
          X_test_scaled,
          reloaded_test
      ))

print("Training float32 preserved:",
      reloaded_train.dtype == np.float32)

print("Testing float32 preserved:",
      reloaded_test.dtype == np.float32)

print("\nSCALER PERSISTENCE VERIFICATION")
print("-" * 60)

print("Scaler features:",
      reloaded_scaler.n_features_in_)

print("Scaler means preserved:",
      np.array_equal(
          scaler.mean_,
          reloaded_scaler.mean_
      ))

print("Scaler variances preserved:",
      np.array_equal(
          scaler.var_,
          reloaded_scaler.var_
      ))

print("Scaler scaling factors preserved:",
      np.array_equal(
          scaler.scale_,
          reloaded_scaler.scale_
      ))

print("Training sample count preserved:",
      np.array_equal(
          scaler.n_samples_seen_,
          reloaded_scaler.n_samples_seen_
      ))

print("\nTraining file size (MB):",
      round(train_scaled_path.stat().st_size / 1e6, 2))

print("Testing file size (MB):",
      round(test_scaled_path.stat().st_size / 1e6, 2))

print("\nPersistence verification completed.")

SCALED DATASET PERSISTENCE VERIFICATION
------------------------------------------------------------

RELOADED TRAINING DATASET
Shape: (2016638, 69)
Data type: float32
Shape correct: True

RELOADED TESTING DATASET
Shape: (504160, 69)
Data type: float32
Shape correct: True

DATA CONSISTENCY VERIFICATION
------------------------------------------------------------
Training values preserved: True
Testing values preserved: True
Training float32 preserved: True
Testing float32 preserved: True

SCALER PERSISTENCE VERIFICATION
------------------------------------------------------------
Scaler features: 69
Scaler means preserved: True
Scaler variances preserved: True
Scaler scaling factors preserved: True
Training sample count preserved: True

Training file size (MB): 556.59
Testing file size (MB): 139.15

Persistence verification completed.


### 18W. Feature Scaling Summary

Feature standardization was successfully completed
using StandardScaler fitted exclusively on the
training dataset.

A batch-based processing approach was implemented
to reduce memory consumption while processing
2,520,798 network traffic records across 69
input features.

The following outcomes were achieved:

- Training records: 2,016,638.
- Testing records: 504,160.
- Input features: 69.
- Training means approximately zero.
- Training standard deviations approximately one.
- No missing or infinite transformed values.
- Original feature ordering preserved.
- Transformed datasets stored in float32 format.
- Fitted scaler and learned parameters preserved.
- Saved datasets successfully reloaded and verified.

The combined standardized dataset storage
requirement is approximately 695.74 MB.

Training-derived scaling parameters were applied
to both datasets without refitting on the
testing dataset, preventing data leakage
through the standardization process.

The original unscaled feature matrix remains
available for tree-based machine learning
models and explainability analysis.

This preprocessing approach supports
reproducibility, computational efficiency,
methodological transparency and subsequent
machine learning experimentation.

## 19. Final Model Preparation

This stage prepares the preprocessed network
traffic datasets for subsequent machine learning
model development and evaluation.

The training and testing datasets are examined
to verify feature-target alignment, dimensional
consistency, class representation and the
availability of preprocessing artifacts.

Both standardized and unscaled feature
representations are retained to support
different machine learning algorithms.

Particular attention is given to preserving
minority attack categories, maintaining
training-testing separation and ensuring
experimental reproducibility.

The final prepared datasets will support
intrusion detection, comparative model
evaluation and subsequent explainable AI
analysis.

### 19B. Feature–Target Alignment Verification

The prepared training and testing datasets are
examined to verify consistency between the
feature matrices and their corresponding
target labels.

This verification ensures that the number of
feature records matches the number of target
labels and that the expected input dimensions
are preserved.

Correct feature-target alignment is essential
for reliable supervised machine learning,
accurate model evaluation and meaningful
explainable AI analysis.

In [23]:
%whos

Variable                Type              Data/Info
---------------------------------------------------
LabelEncoder            type              <class 'sklearn.preproces<...>ing._label.LabelEncoder'>
Path                    type              <class 'pathlib.Path'>
RANDOM_STATE            int               42
StandardScaler          type              <class 'sklearn.preproces<...>ng._data.StandardScaler'>
X                       DataFrame         Shape: (2520798, 69)
X_test_scaled           memmap            [[-0.45271838 -0.3270295 <...>.3811757\n  -0.36185881]]
X_train_scaled          memmap            [[-0.4541389  -0.47018337<...>.3811757\n  -0.36185881]]
affected                int               0
affected_count          int               2331
after_columns           int               70
batch_indices           ndarray           16638: 16638 elems, type `int64`, 133104 bytes (129.984375 kb)
batch_size              int               100000
before_columns          int              

In [24]:
# Feature-target alignment verification

print("FEATURE-TARGET ALIGNMENT VERIFICATION")
print("-" * 65)

print("\nTRAINING DATASET")
print("Feature shape:", X_train_scaled.shape)
print("Target shape:", y_train.shape)
print("Training indices:", len(train_idx))

print("Feature-target count matches:",
      X_train_scaled.shape[0] == len(y_train))

print("Training feature dimensions correct:",
      X_train_scaled.shape[1] == 69)

print("Training target ordering correct:",
      np.array_equal(y_train, y[train_idx]))


print("\nTESTING DATASET")
print("Feature shape:", X_test_scaled.shape)
print("Target shape:", y_test.shape)
print("Testing indices:", len(test_idx))

print("Feature-target count matches:",
      X_test_scaled.shape[0] == len(y_test))

print("Testing feature dimensions correct:",
      X_test_scaled.shape[1] == 69)

print("Testing target ordering correct:",
      np.array_equal(y_test, y[test_idx]))


print("\nDATASET SEPARATION VERIFICATION")
print("-" * 65)

overlap_count = len(np.intersect1d(train_idx, test_idx))

print("Overlapping indices:", overlap_count)

print("Training-testing separation:",
      overlap_count == 0)

print("Training indices unique:",
      len(np.unique(train_idx)) == len(train_idx))

print("Testing indices unique:",
      len(np.unique(test_idx)) == len(test_idx))

print("Total records preserved:",
      len(train_idx) + len(test_idx) == len(X))

print("Complete index coverage:",
      np.array_equal(
          np.sort(np.concatenate([train_idx, test_idx])),
          np.arange(len(X))
      ))

print("\nFEATURE CONSISTENCY")
print("-" * 65)

print("Original feature count:", X.shape[1])

print("Feature ordering preserved:",
      list(X.columns) == list(input_features))

print("\nAlignment verification completed.")

FEATURE-TARGET ALIGNMENT VERIFICATION
-----------------------------------------------------------------

TRAINING DATASET
Feature shape: (2016638, 69)
Target shape: (2016638,)
Training indices: 2016638
Feature-target count matches: True
Training feature dimensions correct: True
Training target ordering correct: True

TESTING DATASET
Feature shape: (504160, 69)
Target shape: (504160,)
Testing indices: 504160
Feature-target count matches: True
Testing feature dimensions correct: True
Testing target ordering correct: True

DATASET SEPARATION VERIFICATION
-----------------------------------------------------------------
Overlapping indices: 0
Training-testing separation: True
Training indices unique: True
Testing indices unique: True
Total records preserved: True
Complete index coverage: True

FEATURE CONSISTENCY
-----------------------------------------------------------------
Original feature count: 69
Feature ordering preserved: True

Alignment verification completed.


### 19C. Transformed Feature Alignment Verification

The standardized feature matrices are independently
verified against the original feature matrix using
the preserved training and testing indices.

Batch-based verification is employed to confirm
that transformed feature values correspond to
their respective original records.

This verification supports dataset integrity,
correct feature-target alignment and reliable
machine learning experimentation.

In [26]:
# Transformed feature alignment verification

print("TRANSFORMED FEATURE ALIGNMENT VERIFICATION")
print("-" * 65)

def verify_feature_alignment(original, indices,
                             transformed, fitted_scaler,
                             batch_size=100000):

    total = len(indices)

    for start in range(0, total, batch_size):

        end = min(start + batch_size, total)

        batch_indices = indices[start:end]

        original_batch = original.iloc[batch_indices]

        expected_batch = fitted_scaler.transform(
    original_batch
).astype(np.float32)

        actual_batch = transformed[start:end]

        if not np.allclose(
            expected_batch,
            actual_batch,
            rtol=1e-5,
            atol=1e-6
        ):
            print("Alignment mismatch:", start, end)
            return False

        print(f"Verified: {end:,} / {total:,}")

    return True


print("\nTRAINING FEATURE ALIGNMENT")

train_alignment = verify_feature_alignment(
    X,
    train_idx,
    X_train_scaled,
    scaler
)

print("\nTESTING FEATURE ALIGNMENT")

test_alignment = verify_feature_alignment(
    X,
    test_idx,
    X_test_scaled,
    scaler
)

print("\nFINAL ALIGNMENT RESULTS")
print("-" * 65)

print("Training feature alignment:", train_alignment)
print("Testing feature alignment:", test_alignment)

print("Complete feature alignment verified:",
      train_alignment and test_alignment)

TRANSFORMED FEATURE ALIGNMENT VERIFICATION
-----------------------------------------------------------------

TRAINING FEATURE ALIGNMENT
Verified: 100,000 / 2,016,638
Verified: 200,000 / 2,016,638
Verified: 300,000 / 2,016,638
Verified: 400,000 / 2,016,638
Verified: 500,000 / 2,016,638
Verified: 600,000 / 2,016,638
Verified: 700,000 / 2,016,638
Verified: 800,000 / 2,016,638
Verified: 900,000 / 2,016,638
Verified: 1,000,000 / 2,016,638
Verified: 1,100,000 / 2,016,638
Verified: 1,200,000 / 2,016,638
Verified: 1,300,000 / 2,016,638
Verified: 1,400,000 / 2,016,638
Verified: 1,500,000 / 2,016,638
Verified: 1,600,000 / 2,016,638
Verified: 1,700,000 / 2,016,638
Verified: 1,800,000 / 2,016,638
Verified: 1,900,000 / 2,016,638
Verified: 2,000,000 / 2,016,638
Verified: 2,016,638 / 2,016,638

TESTING FEATURE ALIGNMENT
Verified: 100,000 / 504,160
Verified: 200,000 / 504,160
Verified: 300,000 / 504,160
Verified: 400,000 / 504,160
Verified: 500,000 / 504,160
Verified: 504,160 / 504,160

FINAL ALIGNME

### 19D. Class Representation Verification

The encoded target distributions are examined
to verify that all original network traffic
classes remain represented in both the training
and testing datasets.

Particular attention is given to minority attack
categories due to the significant class imbalance
within the CICIDS2017 dataset.

This verification supports multiclass intrusion
detection, minority-class performance evaluation
and subsequent explainable AI analysis.

In [27]:
# Class representation verification

print("CLASS REPRESENTATION VERIFICATION")
print("-" * 65)

original_classes = np.unique(y)
training_classes = np.unique(y_train)
testing_classes = np.unique(y_test)

print("Original classes:", len(original_classes))
print("Training classes:", len(training_classes))
print("Testing classes:", len(testing_classes))

print("\nAll classes preserved in training:",
      np.array_equal(original_classes, training_classes))

print("All classes preserved in testing:",
      np.array_equal(original_classes, testing_classes))

class_verification_records = []

for class_id in original_classes:

    original_count = int(np.sum(y == class_id))
    training_count = int(np.sum(y_train == class_id))
    testing_count = int(np.sum(y_test == class_id))

    class_name = label_encoder.inverse_transform(
        [class_id]
    )[0]

    class_verification_records.append({
        "Class_ID": class_id,
        "Traffic_Class": class_name,
        "Original_Count": original_count,
        "Training_Count": training_count,
        "Testing_Count": testing_count,
        "Records_Preserved":
            original_count == training_count + testing_count,
        "Present_in_Both":
            training_count > 0 and testing_count > 0
    })

class_verification_df = pd.DataFrame(
    class_verification_records
)

print("\nCLASS DISTRIBUTION VERIFICATION")
print("-" * 65)

display(class_verification_df)

print("\nAll class records preserved:",
      class_verification_df["Records_Preserved"].all())

print("All classes present in both datasets:",
      class_verification_df["Present_in_Both"].all())

class_verification_path = (
    results_dir / "02_class_representation_verification.csv"
)

class_verification_df.to_csv(
    class_verification_path,
    index=False
)

print("\nVerification report saved successfully:",
      class_verification_path.exists())

CLASS REPRESENTATION VERIFICATION
-----------------------------------------------------------------
Original classes: 15
Training classes: 15
Testing classes: 15

All classes preserved in training: True
All classes preserved in testing: True

CLASS DISTRIBUTION VERIFICATION
-----------------------------------------------------------------


,Class_ID,Traffic_Class,Original_Count,Training_Count,Testing_Count,Records_Preserved,Present_in_Both
0,0,BENIGN,2095057,1676045,419012,True,True
1,1,Bot,1948,1558,390,True,True
2,2,DDoS,128014,102411,25603,True,True
3,3,DoS GoldenEye,10286,8229,2057,True,True
4,4,DoS Hulk,172846,138277,34569,True,True
5,5,DoS Slowhttptest,5228,4182,1046,True,True
6,6,DoS slowloris,5385,4308,1077,True,True
7,7,FTP-Patator,5931,4745,1186,True,True
8,8,Heartbleed,11,9,2,True,True
9,9,Infiltration,36,29,7,True,True



All class records preserved: True
All classes present in both datasets: True

Verification report saved successfully: True


### 19E. Class Weight Verification

The previously calculated class weights are
independently verified against the encoded
training target distribution.

Class weighting is intended to address the
significant imbalance between majority and
minority network traffic categories.

The verification ensures that all class weights
are finite, positive and consistent with the
balanced class weighting formula.

Only training target labels are used in the
calculation to prevent information leakage
from the testing dataset.

In [28]:
# Class weight verification

print("CLASS WEIGHT VERIFICATION")
print("-" * 65)

training_class_ids = np.unique(y_train)

expected_weights = compute_class_weight(
    class_weight="balanced",
    classes=training_class_ids,
    y=y_train
)

stored_weights = np.array([
    class_weights[class_id]
    for class_id in training_class_ids
])

print("Training classes:", len(training_class_ids))

print("Stored class weights:", len(class_weights))

print("All classes have weights:",
      set(class_weights.keys()) == set(training_class_ids))

print("All weights finite:",
      np.isfinite(stored_weights).all())

print("All weights positive:",
      (stored_weights > 0).all())

print("Weights match training-derived calculation:",
      np.allclose(stored_weights, expected_weights))

weight_verification_df = pd.DataFrame({
    "Class_ID": training_class_ids,
    "Traffic_Class": label_encoder.inverse_transform(
        training_class_ids
    ),
    "Training_Count": [
        np.sum(y_train == class_id)
        for class_id in training_class_ids
    ],
    "Stored_Weight": stored_weights,
    "Expected_Weight": expected_weights,
    "Weight_Verified": np.isclose(
        stored_weights,
        expected_weights
    )
})

print("\nCLASS WEIGHT DISTRIBUTION")
print("-" * 65)

display(weight_verification_df)

print("\nAll class weights verified:",
      weight_verification_df["Weight_Verified"].all())

weight_verification_path = (
    results_dir / "02_class_weight_verification.csv"
)

weight_verification_df.to_csv(
    weight_verification_path,
    index=False
)

print("Verification report saved successfully:",
      weight_verification_path.exists())

CLASS WEIGHT VERIFICATION
-----------------------------------------------------------------
Training classes: 15
Stored class weights: 15
All classes have weights: True
All weights finite: True
All weights positive: True
Weights match training-derived calculation: True

CLASS WEIGHT DISTRIBUTION
-----------------------------------------------------------------


,Class_ID,Traffic_Class,Training_Count,Stored_Weight,Expected_Weight,Weight_Verified
0,0,BENIGN,1676045,0.080214,0.080214,True
1,1,Bot,1558,86.291742,86.291742,True
2,2,DDoS,102411,1.312774,1.312774,True
3,3,DoS GoldenEye,8229,16.337651,16.337651,True
4,4,DoS Hulk,138277,0.972270,0.972270,True
5,5,DoS Slowhttptest,4182,32.147904,32.147904,True
6,6,DoS slowloris,4308,31.207645,31.207645,True
7,7,FTP-Patator,4745,28.333516,28.333516,True
8,8,Heartbleed,9,14938.059259,14938.059259,True
9,9,Infiltration,29,4635.949425,4635.949425,True



All class weights verified: True
Verification report saved successfully: True


### 19F. Final Model Artifact Preparation

The essential model preparation artifacts are
persisted to support reproducible machine
learning experimentation.

These artifacts include encoded target labels,
training and testing indices, feature names,
the fitted label encoder, and training-derived
class weights.

The saved artifacts enable subsequent notebooks
to reconstruct the original dataset partitions,
maintain feature ordering, interpret predicted
class labels, and apply class weighting where
appropriate.

This approach reduces unnecessary preprocessing
repetition and supports experimental consistency
across machine learning models.

In [29]:
# Final model artifact preparation

import json

print("FINAL MODEL ARTIFACT PREPARATION")
print("-" * 65)

# Define artifact paths

artifact_paths = {
    "Training targets":
        results_dir / "02_y_train.npy",

    "Testing targets":
        results_dir / "02_y_test.npy",

    "Training indices":
        results_dir / "02_train_indices.npy",

    "Testing indices":
        results_dir / "02_test_indices.npy",

    "Feature names":
        results_dir / "02_feature_names.json",

    "Label encoder":
        results_dir / "02_label_encoder.joblib",

    "Class weights":
        results_dir / "02_class_weights.joblib",

    "Median replacement values":
        results_dir / "02_replacement_values.joblib"
}

# Save encoded target labels

np.save(
    artifact_paths["Training targets"],
    y_train
)

np.save(
    artifact_paths["Testing targets"],
    y_test
)

# Save dataset partition indices

np.save(
    artifact_paths["Training indices"],
    train_idx
)

np.save(
    artifact_paths["Testing indices"],
    test_idx
)

# Save final feature names in original order

with open(
    artifact_paths["Feature names"],
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        X.columns.tolist(),
        file,
        indent=4
    )

# Save fitted label encoder

joblib.dump(
    label_encoder,
    artifact_paths["Label encoder"]
)

# Save training-derived class weights

joblib.dump(
    class_weights,
    artifact_paths["Class weights"]
)

# Save learned replacement values

joblib.dump(
    replacement_values,
    artifact_paths["Median replacement values"]
)

# Generate artifact inventory

artifact_inventory = []

for artifact_name, artifact_path in artifact_paths.items():

    artifact_inventory.append({
        "Artifact": artifact_name,
        "Filename": artifact_path.name,
        "Exists": artifact_path.exists(),
        "Size_MB": round(
            artifact_path.stat().st_size / 1e6,
            6
        ) if artifact_path.exists() else 0
    })

artifact_inventory_df = pd.DataFrame(
    artifact_inventory
)

display(artifact_inventory_df)

print("\nTotal artifacts:",
      len(artifact_inventory_df))

print("All artifacts saved successfully:",
      artifact_inventory_df["Exists"].all())

FINAL MODEL ARTIFACT PREPARATION
-----------------------------------------------------------------


,Artifact,Filename,Exists,Size_MB
0,Training targets,02_y_train.npy,True,16.133232
1,Testing targets,02_y_test.npy,True,4.033408
2,Training indices,02_train_indices.npy,True,16.133232
3,Testing indices,02_test_indices.npy,True,4.033408
4,Feature names,02_feature_names.json,True,0.001695
5,Label encoder,02_label_encoder.joblib,True,0.000699
6,Class weights,02_class_weights.joblib,True,0.000181
7,Median replacement values,02_replacement_values.joblib,True,0.000277



Total artifacts: 8
All artifacts saved successfully: True


### 19G. Model Artifact Persistence Verification

The saved model preparation artifacts are reloaded
and compared with their original in-memory
representations to verify data integrity.

This verification ensures that encoded target labels,
dataset partition indices, feature ordering, label
encoding, class weights, and learned replacement
values remain consistent after serialization.

Successful verification supports reproducibility
and reliable reuse of the preprocessing artifacts
during subsequent machine learning experimentation.

In [30]:
# Model artifact persistence verification

print("MODEL ARTIFACT PERSISTENCE VERIFICATION")
print("-" * 65)

# Reload target labels

saved_y_train = np.load(
    artifact_paths["Training targets"]
)

saved_y_test = np.load(
    artifact_paths["Testing targets"]
)

# Reload dataset indices

saved_train_idx = np.load(
    artifact_paths["Training indices"]
)

saved_test_idx = np.load(
    artifact_paths["Testing indices"]
)

# Reload feature names

with open(
    artifact_paths["Feature names"],
    "r",
    encoding="utf-8"
) as file:

    saved_feature_names = json.load(file)

# Reload fitted preprocessing artifacts

saved_label_encoder = joblib.load(
    artifact_paths["Label encoder"]
)

saved_class_weights = joblib.load(
    artifact_paths["Class weights"]
)

saved_replacement_values = joblib.load(
    artifact_paths["Median replacement values"]
)

# Verify artifact consistency

artifact_checks = {
    "Training targets preserved":
        np.array_equal(saved_y_train, y_train),

    "Testing targets preserved":
        np.array_equal(saved_y_test, y_test),

    "Training indices preserved":
        np.array_equal(saved_train_idx, train_idx),

    "Testing indices preserved":
        np.array_equal(saved_test_idx, test_idx),

    "Feature names and ordering preserved":
        saved_feature_names == X.columns.tolist(),

    "Label encoder classes preserved":
        np.array_equal(
            saved_label_encoder.classes_,
            label_encoder.classes_
        ),

    "Class weights preserved":
        saved_class_weights == class_weights,

    "Median replacement values preserved":
        saved_replacement_values == replacement_values
}

# Display verification results

for check, status in artifact_checks.items():
    print(f"{check}: {status}")

# Create verification report

artifact_verification_df = pd.DataFrame(
    list(artifact_checks.items()),
    columns=["Verification_Check", "Passed"]
)

artifact_verification_path = (
    results_dir / "02_artifact_verification.csv"
)

artifact_verification_df.to_csv(
    artifact_verification_path,
    index=False
)

print("\nTotal verification checks:",
      len(artifact_checks))

print("All artifacts verified:",
      all(artifact_checks.values()))

print("Verification report saved:",
      artifact_verification_path.exists())

MODEL ARTIFACT PERSISTENCE VERIFICATION
-----------------------------------------------------------------
Training targets preserved: True
Testing targets preserved: True
Training indices preserved: True
Testing indices preserved: True
Feature names and ordering preserved: True
Label encoder classes preserved: True
Class weights preserved: True
Median replacement values preserved: True

Total verification checks: 8
All artifacts verified: True
Verification report saved: True


### 19H. Final Model Readiness Assessment

A final readiness assessment is conducted to
consolidate the outcomes of the data preprocessing
and model preparation pipeline.

The assessment summarizes dataset dimensions,
class representation, training-testing separation,
feature consistency, numerical data quality,
and preprocessing artifact availability.

This provides a documented reference for subsequent
machine learning model development, comparative
performance evaluation, and explainable AI analysis.

In [31]:
# Final model readiness assessment

print("FINAL MODEL READINESS ASSESSMENT")
print("-" * 65)

# Essential preprocessing artifacts

essential_artifacts = list(artifact_paths.values()) + [
    train_scaled_path,
    test_scaled_path,
    scaler_path
]

# Consolidated dataset summary

readiness_summary = {
    "Total records": len(X),
    "Training records": len(y_train),
    "Testing records": len(y_test),
    "Input features": X.shape[1],
    "Total traffic classes": len(np.unique(y)),
    "Training classes": len(np.unique(y_train)),
    "Testing classes": len(np.unique(y_test)),
    "Training percentage": round(
        len(y_train) / len(X) * 100, 2
    ),
    "Testing percentage": round(
        len(y_test) / len(X) * 100, 2
    ),
    "Random state": RANDOM_STATE,
    "Scaled training data type": str(
        X_train_scaled.dtype
    ),
    "Scaled testing data type": str(
        X_test_scaled.dtype
    )
}

readiness_summary_df = pd.DataFrame(
    list(readiness_summary.items()),
    columns=["Metric", "Value"]
)

display(readiness_summary_df)

# Final readiness checks

readiness_checks = {
    "Training feature-target alignment":
        X_train_scaled.shape[0] == len(y_train),

    "Testing feature-target alignment":
        X_test_scaled.shape[0] == len(y_test),

    "Training feature count correct":
        X_train_scaled.shape[1] == X.shape[1],

    "Testing feature count correct":
        X_test_scaled.shape[1] == X.shape[1],

    "Training-testing separation":
        np.intersect1d(train_idx, test_idx).size == 0,

    "All records preserved":
        len(y_train) + len(y_test) == len(X),

    "All classes represented in training":
        np.array_equal(
            np.unique(y_train), np.unique(y)
        ),

    "All classes represented in testing":
        np.array_equal(
            np.unique(y_test), np.unique(y)
        ),

    "Training data free of missing values":
        train_nan == 0,

    "Testing data free of missing values":
        test_nan == 0,

    "Training data free of infinite values":
        train_inf == 0,

    "Testing data free of infinite values":
        test_inf == 0,

    "Model preparation artifacts verified":
        all(artifact_checks.values()),

    "All essential artifacts available":
        all(path.exists() for path in essential_artifacts)
}

readiness_checks_df = pd.DataFrame(
    list(readiness_checks.items()),
    columns=["Readiness_Check", "Passed"]
)

print("\nFINAL READINESS CHECKS")
print("-" * 65)

display(readiness_checks_df)

# Save final reports

readiness_summary_df.to_csv(
    results_dir / "02_final_dataset_summary.csv",
    index=False
)

readiness_checks_df.to_csv(
    results_dir / "02_final_readiness_checks.csv",
    index=False
)

print("\nTotal readiness checks:",
      len(readiness_checks))

print("Checks passed:",
      sum(readiness_checks.values()))

print("Essential artifacts:",
      len(essential_artifacts))

print("Model preparation checks passed:",
      all(readiness_checks.values()))

print("Final readiness reports saved successfully.")

FINAL MODEL READINESS ASSESSMENT
-----------------------------------------------------------------


,Metric,Value
0,Total records,2520798
1,Training records,2016638
2,Testing records,504160
3,Input features,69
4,Total traffic classes,15
5,Training classes,15
6,Testing classes,15
7,Training percentage,80.0
8,Testing percentage,20.0
9,Random state,42



FINAL READINESS CHECKS
-----------------------------------------------------------------


,Readiness_Check,Passed
0,Training feature-target alignment,True
1,Testing feature-target alignment,True
2,Training feature count correct,True
3,Testing feature count correct,True
4,Training-testing separation,True
5,All records preserved,True
6,All classes represented in training,True
7,All classes represented in testing,True
8,Training data free of missing values,True
9,Testing data free of missing values,True



Total readiness checks: 14
Checks passed: 14
Essential artifacts: 11
Model preparation checks passed: True
Final readiness reports saved successfully.


## 20. Final Preprocessing Conclusion

This notebook successfully established a structured
and reproducible data preprocessing pipeline for
machine learning-based intrusion detection using
the CICIDS2017 dataset.

### 20.1 Final Dataset Characteristics

| Metric | Value |
|---|---:|
| Total network traffic records | 2,520,798 |
| Training records | 2,016,638 |
| Testing records | 504,160 |
| Input features | 69 |
| Traffic classes | 15 |
| Training proportion | 80% |
| Testing proportion | 20% |
| Random state | 42 |
| Standardized feature data type | float32 |
| Essential preprocessing artifacts | 11 |
| Final readiness checks passed | 14/14 |

### 20.2 Major Preprocessing Achievements

The following preprocessing activities were
successfully completed:

- Dataset loading and structural inspection.
- Feature consistency and redundancy assessment.
- Numerical data quality assessment.
- Negative and invalid value analysis.
- Invalid value treatment using training-derived
  replacement values.
- Feature and target separation.
- Multiclass label encoding.
- Stratified training-testing partitioning.
- Class imbalance analysis.
- Training-derived class weight computation.
- Memory-efficient batch-based feature standardization.
- Scaled dataset persistence and verification.
- Feature-target alignment verification.
- Minority class preservation verification.
- Class weight verification.
- Model preparation artifact persistence.
- Final model readiness assessment.

### 20.3 Methodological Considerations

The standardization parameters and replacement
values were derived from the training dataset
to reduce the risk of preprocessing-related
data leakage.

All 15 traffic classes were preserved in both
the training and testing datasets.

The standardized feature matrices were stored
using the float32 data type and memory-mapped
processing to improve computational efficiency.

The preprocessing artifacts were saved and
verified to support reproducibility and
consistent experimental conditions.

### 20.4 Research Limitations

Despite successful preprocessing, several
limitations remain relevant to subsequent
machine learning experimentation:

- Severe class imbalance across traffic categories.
- Extremely limited representation of rare attacks,
  particularly Heartbleed, Infiltration and SQL Injection.
- Potential model sensitivity to large class weights.
- Limited generalizability of results obtained from
  a single benchmark dataset.
- Potential dataset-specific correlations and
  residual sources of data leakage.
- Computational constraints associated with
  large-scale machine learning experimentation.

These limitations will be considered during
model development, evaluation and interpretation.

### 20.5 Transition to Machine Learning Development

The prepared datasets and verified preprocessing
artifacts provide the foundation for subsequent
machine learning experimentation.

The next stage will focus on baseline model
development, comparative performance evaluation,
class imbalance mitigation and the identification
of suitable models for explainable AI analysis.

Particular attention will be given to minority
attack detection, false positive rates,
class-specific performance and model interpretability.

**Notebook Status: Completed — Data Preprocessing
and Model Preparation.**